### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="ecoli_proteins",
    dataset_year="1996",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5388M",
    download_description="""

wget https://archive.ics.uci.edu/static/public/39/ecoli.zip && unzip ecoli.zip && rm ecoli.zip ecoli.names
mkdir -p local-data-warehouse/ecoli_proteins && mv ecoli.data local-data-warehouse/ecoli_proteins/
""",
    # References
    academic_reference_bibtex="""@inproceedings{horton1996probabilistic,
  title={A probabilistic classification system for predicting the cellular localization sites of proteins.},
  author={Horton, Paul and Nakai, Kenta},
  booktitle={Ismb},
  volume={4},
  pages={109--115},
  year={1996},
  organization={St. Louis, Missouri, USA}
}
""",
    academic_reference_bibtex_key="horton1996probabilistic",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
We start with the dataset from UCI.

- We drop the sequence name column, which is a unique identifier.
- We drop three classes with less than 10 samples each: omL (5), imL (2), and imS (3).
- We drop "chg", which becomes constant after our preprocessing steps.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="class",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="class",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

feature_names = [
    "Sequence Name",
    "mcg",
    "gvh",
    "lip",
    "chg",
    "aac",
    "alm1",
    "alm2",
    "class",
]
df = pd.read_csv(dataset_mold.path / "ecoli.data", header=None, names=feature_names, sep="\s+")
print("Loaded data shape:", df.shape)

df = df.drop(columns=[
    "Sequence Name",  # avoid leakage from look the name and otherwise unique identifier.
    "chg", # constant after preprocessing
])

# Drop samples with classes with too little data (less than 10 samples)
df = df[~df["class"].isin(["omL", "imL", "imS"])]

as_cat_type = ["lip", "class"]
df[as_cat_type] = df[as_cat_type].astype("category")


df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (336, 9)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 327
Columns: 7
Use sampling: False (sample size: 327)
Get row duplicates (staged, merged)...
Using top-6 columns for initial filtering: ['alm1', 'mcg', 'alm2', 'gvh', 'aac', 'lip']
Rows remaining as candidates after top-6 filter: 0 (of 327)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,mcg,gvh,lip,aac,alm1,alm2,class
0,0.76,0.71,0.48,0.50,0.71,0.75,imU
1,0.32,0.33,0.48,0.60,0.06,0.20,cp
2,0.63,0.51,0.48,0.64,0.72,0.76,imU
3,0.42,0.40,0.48,0.56,0.18,0.30,cp
4,0.30,0.45,0.48,0.36,0.21,0.32,cp


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,lip,category,0.0,0.0,2.0,"0.48, 1.0"
1,class,category,0.0,0.0,5.0,"cp, im, pp, imU, om"
2,mcg,float64,0.0,0.0,77.0,"0.63, 0.34, 0.44, 0.64, 0.74, 0.29, 0.4, 0.67, 0.69, 0.25"
3,gvh,float64,0.0,0.0,63.0,"0.51, 0.4, 0.42, 0.47, 0.37, 0.49, 0.46, 0.57, 0.44, 0.39"
4,aac,float64,0.0,0.0,59.0,"0.42, 0.46, 0.48, 0.51, 0.41, 0.49, 0.57, 0.43, 0.56, 0.54"
5,alm1,float64,0.0,0.0,81.0,"0.35, 0.39, 0.28, 0.33, 0.78, 0.76, 0.71, 0.38, 0.45, 0.49"
6,alm2,float64,0.0,0.0,75.0,"0.39, 0.43, 0.35, 0.74, 0.41, 0.33, 0.38, 0.79, 0.44, 0.42"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
mcg,327.0,0.494190,0.193719,0.00,0.89
gvh,327.0,0.499939,0.149935,0.16,1.00
aac,327.0,0.499450,0.123165,0.00,0.88
alm1,327.0,0.497462,0.217333,0.03,1.00
alm2,327.0,0.503119,0.206947,0.00,0.99


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                    
class  1       cp    143  43.73
       2       im     77  23.55
       3       pp     52  15.90
       4      imU     35  10.70
       5       om     20   6.12
lip    1     0.48    324  99.08
       2      1.0      3   0.92

In [8]:
# Target Distribution
target_df

,count,pct
class,,
cp,143,43.73
im,77,23.55
pp,52,15.90
imU,35,10.70
om,20,6.12


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to ecoli_proteins/019d736a-d6d2-7e28-aeff-7da6b7126b9d


019d736a-d6d2-7e28-aeff-7da6b7126b9d
886458d37ea9b8b8987d67d68eed72ea032f098b0d8eee99f44e7e262a9f09d0
